# KDIC 검색 평가 — Jina Reranker v2 A/B (Colab)

로컬 `crawler/` 구조(`rag.py` + `eval_anchor_v3.py` + `gt_relabeled_v7.jsonl`)를 그대로 Colab에서 돌리기 위한 노트북입니다.

**이 노트북이 로컬과 다르게 하는 것 2가지 (설계상 의도한 조정 — 문제 아님):**
1. **인덱스 파일(`faiss.index`/`bm25.pkl`/`emb.npy`)을 업로드하지 않고, `chunks.jsonl`로부터 Colab에서 새로 빌드합니다.** 피클/faiss 인덱스는 로컬(Windows) 환경에서 만들어진 바이너리라 Colab(Linux)에서 버전 호환 문제가 생길 수 있어서, 코드(`rag.py`의 `Store`)로 재생성하는 쪽이 더 안전합니다. 어차피 코퍼스가 215개 청크뿐이라 재빌드는 몇 초 안에 끝납니다.
2. **데이터 파일(`chunks.jsonl`, `gt_relabeled_v7.jsonl`)은 업로드 위젯으로 직접 올려야 합니다.** 코드 파일(`rag.py`, `eval_anchor_v3.py`)은 아래 셀에 그대로 박아뒀지만, 데이터는 용량이 있어서(각각 ~340KB/~1.1MB) 노트북에 인라인으로 넣지 않았습니다.

## 실행 순서
1. (선택) 런타임 유형을 GPU로 바꾸면 리랭커가 더 빠릅니다 — 상단 메뉴 `런타임 > 런타임 유형 변경`. CPU로도 동작은 합니다(215개 청크 규모라 느리지 않음).
2. 아래 셀을 위에서부터 순서대로 실행.
3. "데이터 업로드" 셀에서 로컬의 `data/chunks.jsonl`과 `data/gt_relabeled_v7.jsonl` 두 파일을 선택해서 올려주세요.


## 1. 의존성 설치

In [ ]:
!pip install -q faiss-cpu "sentence-transformers==3.3.1" "transformers==4.46.3" kiwipiepy rank_bm25 einops


**참고 1**: `transformers==4.46.3`으로 고정한 이유 — Jina reranker v2(`jinaai/jina-reranker-v2-base-multilingual`)의 커스텀 원격 코드(`trust_remote_code=True`로 실행되는 `embedding.py`)가 `transformers.models.xlm_roberta.modeling_xlm_roberta.create_position_ids_from_input_ids`를 직접 import합니다. Colab 기본 최신 `transformers`에서는 이 내부 함수가 삭제/이동돼서 `ImportError`가 납니다. `4.46.3`은 로컬 개발 환경에서 이미 검증된 버전이라 이걸로 고정했습니다.

**만약 위 pip 셀을 이미 건너뛰고 최신 transformers로 한 번 실행해봤다면**: 이 셀을 다시 실행해도 세션 메모리엔 예전 버전이 남아있어서 효과가 없습니다 — 반드시 `런타임 > 세션 다시 시작` 후 1번 셀부터 재실행하세요.

**참고 2**: `kiwipiepy`(한국어 형태소 분석기, BM25 토큰화용)는 Colab의 Linux 환경에서 PyPI 사전빌드 wheel로 설치되는 게 일반적이지만, 로컬(Windows)에서 직접 실행/검증은 못 했습니다. 이 셀에서 에러가 나면 알려주세요 — 대체 토크나이저로 바꾸는 방법을 찾아드리겠습니다.

## 2. 디렉터리 구조 준비 (로컬과 동일하게: `data/`, `data/index/`)

In [ ]:
import os
os.makedirs("data/index", exist_ok=True)
print("ok")


## 3. 코드 파일 생성 (`rag.py`, `eval_anchor_v3.py`)

로컬 파일 내용을 그대로 씁니다 — 리랭커 관련 수정분(`get_reranker`, `Searcher.rerank`, `hybrid_bf_rerank` 모드, `BF_DOC_CAP=5`) 포함된 최신 버전입니다.

In [ ]:
%%writefile rag.py
"""RAG 검색 공용 모듈 — 임베딩 · dense(FAISS) · BM25(kiwi) · 하이브리드(RRF).

- 임베딩: sentence-transformers `jhgan/ko-sroberta-multitask` (로컬 한국어 SBERT, 768d).
- dense: FAISS IndexFlatIP + 정규화 임베딩(= 코사인).
- sparse: rank_bm25 BM25Okapi, kiwipiepy 형태소 토큰.
- 융합: RRF(k=60). 리랭킹·파라미터 튜닝은 D1 범위 밖(D2 판단).

Store: doc_id 단위 upsert + content_hash 스킵 (recollect 재수집 트리거용).
"""

from __future__ import annotations

import json
import pathlib
import pickle

import faiss
import numpy as np

MODEL_NAME = "dragonkue/BGE-m3-ko"  # M2.5(항목13): base bge-m3 대비 벤치용 교체(2026-08-02)
INDEX_DIR = "data/index"
EMB_DIM = 1024   # bge-m3 계열 공통 차원(모델 교체 시 갱신). build 시 실제 임베딩 차원으로 자동 보정됨.
RERANK_MODEL_NAME = "jinaai/jina-reranker-v2-base-multilingual"
RERANK_POOL = 30   # 리랭커에 넘길 1차 후보 개수(cap 적용 후 top-N)

_model = None
_kiwi = None
_reranker = None


def get_model():
    global _model
    if _model is None:
        from sentence_transformers import SentenceTransformer
        _model = SentenceTransformer(MODEL_NAME)
    return _model


def get_reranker():
    """Jina reranker v2(cross-encoder) 지연 로드. trust_remote_code 필요(커스텀 모델링 코드)."""
    global _reranker
    if _reranker is None:
        from sentence_transformers import CrossEncoder
        _reranker = CrossEncoder(RERANK_MODEL_NAME, trust_remote_code=True)
    return _reranker


def _get_kiwi():
    global _kiwi
    if _kiwi is None:
        from kiwipiepy import Kiwi
        _kiwi = Kiwi()
    return _kiwi


def tokenize(text: str) -> list[str]:
    """BM25용 한국어 토큰 (조사·기호 제거, 2자+ 명사/어간 위주)."""
    kiwi = _get_kiwi()
    keep = {"NNG", "NNP", "NNB", "NR", "NP", "VV", "VA", "SL", "SN", "SH", "MAG"}
    out = []
    for t in kiwi.tokenize(text):
        if t.tag in keep and len(t.form) >= 1:
            out.append(t.form.lower())
    return out or [text.lower()]


def embed_texts(texts: list[str]) -> np.ndarray:
    m = get_model()
    e = m.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    return np.asarray(e, dtype="float32")


def chunk_embed_text(chunk: dict) -> str:
    """검색 임베딩 입력 — 제목 맥락을 본문 앞에 덧붙임."""
    title = chunk.get("page_title", "")
    return f"{title}\n{chunk['text']}" if title else chunk["text"]


# ── 질의 업무 분류 (하드 필터용, 결정론적 키워드) ─────────────────────────
# chunk.py 문서측 재태깅과 짝을 이루는 질의측 분류. 사용자 구어체 어휘 반영.
# 점수/마진 미달(불명)이면 None → 필터 미적용(전체 검색)으로 안전 폴백.
BF_QUERY_KEYWORDS = {
    "예금자보호제도": [("보호돼", 3), ("보호되", 3), ("보호받", 3), ("보호 되", 3),
                  ("보호대상", 3), ("보호한도", 3), ("예금자보호", 3), ("예금 보호", 3),
                  ("얼마까지", 2), ("한도", 2), ("1억", 2), ("5천", 2), ("새마을금고", 3),
                  ("신협", 3), ("우체국", 2), ("외화", 2), ("펀드", 2),
                  ("예금보험 가입", 2), ("세전", 2), ("저축은행", 1)],
    "예금보험금 안내": [("예금보험금", 3), ("가지급금", 3), ("개산지급금", 3), ("보험금", 2),
                  ("보험사고", 2), ("망하면", 2), ("문 닫으면", 2), ("대신 주는", 2),
                  ("대신 준다", 2), ("지급", 1), ("창구", 1)],
    "고객 미수령금 신청": [("미수령", 3), ("못 찾", 3), ("못 받은 돈", 3), ("안 찾아간", 3),
                   ("찾아가지", 3), ("돌아가신", 2), ("상속인", 2), ("상속", 1), ("통합신청", 2)],
    "착오송금 반환 신청": [("착오송금", 3), ("잘못 보낸", 3), ("잘못 송금", 3), ("잘못 이체", 3),
                   ("이체 실수", 3), ("계좌 잘못", 3), ("돈 잘못", 3), ("딴 사람", 2),
                   ("반환", 2), ("수취인", 2), ("이체수수료", 2), ("이의제기", 2)],
    "채무조정 안내": [("채무조정", 3), ("개인회생", 3), ("면책", 3), ("신용회복", 3),
                 ("빚", 3), ("채무", 3), ("감면", 3), ("빚 조정", 3), ("빚 감면", 3),
                 ("파산", 2), ("연체", 2)],
    "은닉재산 신고": [("은닉재산", 3), ("숨긴 재산", 3), ("숨겨둔", 3), ("숨긴 돈", 3),
                 ("제보", 3), ("포상금", 3), ("부실관련자", 3), ("신고센터", 2),
                 ("은닉", 2), ("신고", 1)],
}
QBF_MIN_SCORE = 3
QBF_MARGIN = 2
BF_DOC_CAP = 5     # hybrid_bf: parent_doc_id당 top-k 청크 수 제한(문서 다양성↑, 몬스터 문서 독점 차단)
# 업무 신호 반영 방식: "soft"(같은 업무 청크에 RRF +BF_BOOST, 제외 없음) | "hard"(타 업무 제외).
# soft 채택(2026-07-22): 오분류 시 정답을 배제하지 않아 실트래픽에 안전 + 리랭킹 토대. 하드는 A/B 재현용 보존.
BF_MODE = "soft"
# bge-m3 재튜닝(2026-07-22): 강한 dense에 맞춰 부스트↓·dense가중↑ (eval_rrf_tune A/B, A안).
BF_BOOST = 0.005          # 소프트부스트 강도(과보정 방지: 0.02→0.005)
RRF_ALPHA = 0.6        # 가중 RRF의 dense 비중(0.5 동일가중~1.0 dense-only). sparse=1-RRF_ALPHA
RRF_K = 5                 # RRF 상수(낮을수록 상위 랭크 신뢰↑). bge-m3 스윕 최적값(60→5).
FACET_BOOST = 0.03        # 서브인텐트 패싯 부스트(업무 내부 형제문서 구분). sub_category 매칭 청크에 가산. eval_facet A/B 채택.
# 업무 공용 문서: 하드필터 시 해당 업무 외에도 허용할 문서(태그는 1개지만 실제로 여러 업무에서 필요).
# 예: 예금보험금 구비서류(DpsmIbamtAplyPossDcmnt)는 미수령금 신청 구비서류로도 동일하게 쓰임.
BF_SHARED_DOCS = {
    "고객 미수령금 신청": {"kdic-www-sp-dpstrprot-DpsmIbamtAplyPossDcmnt-selectScrn"},
}


def classify_query_bf(query: str):
    """질의 → 6대 업무 중 하나 또는 None(불명). 하드 필터의 대상 업무 결정.

    공백 정규화: 질의·키워드 양쪽 공백 제거 후 매칭 → "보호 한도"가 "보호한도"로 잡힘.
    """
    q = query.replace(" ", "")
    sc = {bf: sum(w * q.count(k.replace(" ", "")) for k, w in kws)
          for bf, kws in BF_QUERY_KEYWORDS.items()}
    ranked = sorted(sc.items(), key=lambda x: -x[1])
    if ranked[0][1] < QBF_MIN_SCORE:
        return None
    if len(ranked) > 1 and ranked[0][1] - ranked[1][1] < QBF_MARGIN:
        return None
    return ranked[0][0]


# ── 서브인텐트 패싯(업무 내부 문서타입) 분류 — hybrid_soft 패싯 부스트용 ──────────
# 질의 인텐트를 결정론 키워드로 잡아, 매칭되는 sub_category 청크에 소량 가산.
# 문서 병합/창작 없이 기존 sub_category 패싯만 활용. 리랭킹 대체 경량 레버.
FACET_RULES = [
    (["서류", "구비", "준비물", "챙기", "제출", "들고", "가지고"], "구비서류"),
    (["신청방법", "어떻게 신청", "어떻게 해", "온라인", "인터넷", "접수"], "신청방법"),
    (["대상", "자격", "누가", "해당", "받을 수 있"], "신청대상"),
    (["절차", "단계", "처리", "과정", "진행"], "절차"),
    (["유의", "주의", "조심", "알아둘"], "유의사항"),
    (["방문", "직접 가", "찾아가", "가서"], "방문접수"),
    (["법령", "규정", "법적"], "관련법령"),
]
DIR_RULES = [
    (["받은", "들어온", "잘못 들어", "수취인", "받아버린"], "착오송금수취인"),
    (["보낸", "송금한", "이체한", "내가 보", "잘못 보", "잘못 송금"], "착오송금인"),
]


def query_facets(query: str) -> set:
    """질의 → sub_category에서 찾을 패싯 토큰 집합(문서 타입/방향)."""
    qn = query.replace(" ", "")
    facets = set()
    for kws, tok in FACET_RULES:
        if any(k.replace(" ", "") in qn for k in kws):
            facets.add(tok)
    for kws, tok in DIR_RULES:
        if any(k.replace(" ", "") in qn for k in kws):
            facets.add(tok); break
    return facets


class Store:
    """청크 + 임베딩 + 인덱스 저장소. doc_id 단위 upsert 지원."""

    def __init__(self, index_dir: str = INDEX_DIR):
        self.dir = pathlib.Path(index_dir)
        self.chunks: list[dict] = []
        self.embs: np.ndarray | None = None
        self.doc_hash: dict[str, str] = {}
        self._load_state()

    # ── 상태 로드/저장 ───────────────────────────────────────
    def _load_state(self) -> None:
        meta = self.dir / "chunk_meta.jsonl"
        emb = self.dir / "emb.npy"
        docs = self.dir / "docs.json"
        if meta.exists() and emb.exists():
            self.chunks = [json.loads(l) for l in meta.read_text(encoding="utf-8").splitlines() if l]
            self.embs = np.load(emb)
        if docs.exists():
            self.doc_hash = json.loads(docs.read_text(encoding="utf-8"))

    def needs_update(self, doc_id: str, content_hash: str) -> bool:
        return self.doc_hash.get(doc_id) != content_hash

    # ── upsert / remove (doc_id 단위) ────────────────────────
    def upsert(self, doc_id: str, chunks: list[dict], content_hash: str) -> None:
        self._drop(doc_id)
        if chunks:
            e = embed_texts([chunk_embed_text(c) for c in chunks])
            self.chunks.extend(chunks)
            self.embs = e if self.embs is None else np.vstack([self.embs, e])
        self.doc_hash[doc_id] = content_hash

    def remove(self, doc_id: str) -> None:
        self._drop(doc_id)
        self.doc_hash.pop(doc_id, None)

    def _drop(self, doc_id: str) -> None:
        keep = [i for i, c in enumerate(self.chunks) if c["parent_doc_id"] != doc_id]
        if len(keep) != len(self.chunks):
            self.chunks = [self.chunks[i] for i in keep]
            if self.embs is not None and len(self.embs):
                self.embs = self.embs[keep] if keep else None

    # ── 인덱스 빌드 + 영속화 ─────────────────────────────────
    def build_and_save(self) -> None:
        self.dir.mkdir(parents=True, exist_ok=True)
        n = len(self.chunks)
        embs = self.embs if self.embs is not None else np.zeros((0, EMB_DIM), "float32")

        # 인덱스 차원 = 실제 임베딩 차원(모델 교체 대응). 비었으면 EMB_DIM 폴백.
        dim = int(embs.shape[1]) if embs.shape[0] else EMB_DIM
        index = faiss.IndexFlatIP(dim)
        if n:
            index.add(embs)
        faiss.write_index(index, str(self.dir / "faiss.index"))
        np.save(self.dir / "emb.npy", embs)

        from rank_bm25 import BM25Okapi
        corpus = [tokenize(c["text"]) for c in self.chunks]
        bm25 = BM25Okapi(corpus) if corpus else None
        with open(self.dir / "bm25.pkl", "wb") as f:
            pickle.dump({"bm25": bm25, "corpus_len": n}, f)

        with open(self.dir / "chunk_meta.jsonl", "w", encoding="utf-8") as f:
            for c in self.chunks:
                f.write(json.dumps(c, ensure_ascii=False) + "\n")
        (self.dir / "docs.json").write_text(
            json.dumps(self.doc_hash, ensure_ascii=False, indent=2), encoding="utf-8")

        # chunks.jsonl 도 최신화 (T3 산출물과 일치)
        with open("data/chunks.jsonl", "w", encoding="utf-8") as f:
            for c in self.chunks:
                f.write(json.dumps(c, ensure_ascii=False) + "\n")


class Searcher:
    """dense / bm25 / hybrid(RRF) 검색."""

    def __init__(self, index_dir: str = INDEX_DIR):
        self.dir = pathlib.Path(index_dir)
        self.chunks = [json.loads(l) for l in
                       (self.dir / "chunk_meta.jsonl").read_text(encoding="utf-8").splitlines() if l]
        self.index = faiss.read_index(str(self.dir / "faiss.index"))
        with open(self.dir / "bm25.pkl", "rb") as f:
            self.bm25 = pickle.load(f)["bm25"]

    def dense(self, query: str, k: int = 10) -> list[tuple[int, float]]:
        q = embed_texts([query])
        D, I = self.index.search(q, min(k, len(self.chunks)))
        return [(int(i), float(d)) for i, d in zip(I[0], D[0]) if i >= 0]

    def sparse(self, query: str, k: int = 10) -> list[tuple[int, float]]:
        if self.bm25 is None:
            return []
        scores = self.bm25.get_scores(tokenize(query))
        order = np.argsort(scores)[::-1][:k]
        return [(int(i), float(scores[i])) for i in order]

    def hybrid(self, query: str, k: int = 10, pool: int = 20,
               rrf_k: int = RRF_K, bf: str | None = None) -> list[tuple[int, float]]:
        # bf 지정 시(하드 필터): 후보를 해당 업무 청크로 제한. 필터링으로 후보가
        # 줄므로 pool을 전체로 넓혀 in-bf 청크를 빠짐없이 확보.
        dpool = pool if bf is None else len(self.chunks)
        d = self.dense(query, dpool)
        s = self.sparse(query, dpool)
        if bf is not None:
            allow = BF_SHARED_DOCS.get(bf, set())
            def _keep(i):
                c = self.chunks[i]
                return c.get("business_function") == bf or c["parent_doc_id"] in allow
            d = [(i, sc) for i, sc in d if _keep(i)]
            s = [(i, sc) for i, sc in s if _keep(i)]
        rrf: dict[int, float] = {}
        for rank, (idx, _) in enumerate(d):
            rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (rrf_k + rank + 1)
        for rank, (idx, _) in enumerate(s):
            rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (rrf_k + rank + 1)
        ranked = sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:k]
        return [(idx, score) for idx, score in ranked]

    def hybrid_soft(self, query: str, k: int = 10, rrf_k: int = RRF_K,
                    bf: str | None = None, boost: float = 0.0,
                    alpha: float = RRF_ALPHA) -> list[tuple[int, float]]:
        """소프트부스트 융합 — 전체 풀 RRF 후 같은 업무 청크에 +boost(제외 없음).

        하드필터(hybrid(bf=...))와 달리 타 업무 청크를 남겨두므로, 질의 분류가
        틀려도 정답이 후보에 살아있다(graceful degradation). boost=0이면 순수 하이브리드.
        """
        N = len(self.chunks)
        d = self.dense(query, N)
        s = self.sparse(query, N)
        rrf: dict[int, float] = {}
        for rank, (idx, _) in enumerate(d):
            rrf[idx] = rrf.get(idx, 0.0) + alpha / (rrf_k + rank + 1)
        for rank, (idx, _) in enumerate(s):
            rrf[idx] = rrf.get(idx, 0.0) + (1.0 - alpha) / (rrf_k + rank + 1)
        if bf is not None and boost > 0:
            allow = BF_SHARED_DOCS.get(bf, set())
            facets = query_facets(query) if FACET_BOOST > 0 else set()
            for idx in rrf:
                c = self.chunks[idx]
                if c.get("business_function") == bf or c["parent_doc_id"] in allow:
                    rrf[idx] += boost
                    if facets:
                        hit = sum(1 for t in facets if t in c.get("sub_category", ""))
                        if hit:
                            rrf[idx] += FACET_BOOST * hit
        ranked = sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:k]
        return ranked

    def rerank(self, query: str, candidates: list[tuple[int, float]], k: int) -> list[tuple[int, float]]:
        """cross-encoder로 후보(candidates)만 재정렬 — 전체 코퍼스가 아니라 1차 검색 결과만 대상."""
        if not candidates:
            return candidates
        reranker = get_reranker()
        pairs = [(query, chunk_embed_text(self.chunks[idx])) for idx, _ in candidates]
        scores = reranker.predict(pairs)
        reranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
        return [(idx, float(sc)) for (idx, _), sc in reranked[:k]]

    def _cap_by_doc(self, hits: list[tuple[int, float]], cap: int) -> list[tuple[int, float]]:
        """parent_doc_id당 cap개까지만 순서 유지하며 남김(문서 단위 다양성 확보)."""
        if not cap or cap <= 0:
            return hits
        seen: dict[str, int] = {}
        out = []
        for idx, score in hits:
            d = self.chunks[idx]["parent_doc_id"]
            if seen.get(d, 0) < cap:
                out.append((idx, score))
                seen[d] = seen.get(d, 0) + 1
        return out

    def search(self, query: str, k: int = 5, mode: str = "hybrid") -> list[dict]:
        if mode == "dense":
            hits = self.dense(query, k)
        elif mode == "bm25":
            hits = self.sparse(query, k)
        elif mode == "hybrid_bf":
            # 질의 업무 분류 → 업무 신호 반영(soft 부스트 / hard 제외) + doc-count cap.
            # 깊은 풀을 받아 문서당 BF_DOC_CAP개로 제한 후 top-k → 몬스터 문서 독점 차단.
            bf = classify_query_bf(query)
            if BF_MODE == "soft":
                pool = self.hybrid_soft(query, len(self.chunks), bf=bf, boost=BF_BOOST)
            else:
                pool = self.hybrid(query, max(k * 5, 50), bf=bf)
            hits = self._cap_by_doc(pool, BF_DOC_CAP)[:k]
        elif mode == "hybrid_bf_rerank":
            # hybrid_bf와 동일하게 1차 후보를 뽑되, cap 적용 후 top-k로 자르지 않고
            # RERANK_POOL개까지 넉넉히 남긴 뒤 cross-encoder로 재정렬해서 top-k만 반환.
            bf = classify_query_bf(query)
            if BF_MODE == "soft":
                pool = self.hybrid_soft(query, len(self.chunks), bf=bf, boost=BF_BOOST)
            else:
                pool = self.hybrid(query, max(RERANK_POOL * 2, 50), bf=bf)
            capped = self._cap_by_doc(pool, BF_DOC_CAP)[:RERANK_POOL]
            hits = self.rerank(query, capped, k)
        else:
            hits = self.hybrid(query, k)
        out = []
        for idx, score in hits:
            c = dict(self.chunks[idx])
            c["_score"] = round(score, 4)
            out.append(c)
        return out


In [ ]:
%%writefile eval_anchor_v3.py
"""KDIC RAG v3 청크 단위 검색 평가.

기존 ``eval.py``의 parent_doc_id 기반 평가는 변경하지 않는다. 이 모듈은
GT의 gold chunk와 검색된 chunk_id를 직접 비교하는 별도 평가기다.

사용 예:
    cd crawler
    python eval_anchor_v3.py --retrieved data/retrieved.jsonl
    python eval_anchor_v3.py --mode hybrid_bf
    python eval_anchor_v3.py --mode hybrid_bf_rerank --limit 30   # 속도 측정용 스모크 테스트

``--retrieved``를 주면 검색을 실행하지 않고 JSONL의 결과를 그대로 평가한다.
각 결과 행은 ``id``와 순위순 ``retrieved_chunk_ids``를 가져야 한다.

``--limit N``을 주면 GT의 처음 N건만 평가한다 — 느린 모드(리랭커 등) 속도 확인용이며,
업무가 뒤섞여 있지 않은 순서라 지표 자체는 대표성이 없다(전체 평가로 재확인할 것).
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Iterable


KS = (1, 3, 5, 10)
RECALL_KS = (3, 5, 10)
RETRIEVAL_DEPTH = 10


def _default_data_dir() -> Path:
    """스크립트 위치에서 GT v7이 있는 data 폴더를 찾는다."""
    for parent in Path(__file__).resolve().parents:
        data_dir = parent / "data"
        if (data_dir / "gt_relabeled_v7.jsonl").is_file():
            return data_dir
    raise FileNotFoundError("gt_relabeled_v7.jsonl을 포함한 data 폴더를 찾을 수 없습니다.")


def _get_gt_chunk_ids(item: dict) -> list[str]:
    """새 ``gt_chunk_ids`` 우선, 비어 있으면 구형 단일 gold를 사용한다."""
    chunk_ids = item.get("gt_chunk_ids")
    if chunk_ids:
        if isinstance(chunk_ids, str):
            chunk_ids = [chunk_ids]
        return list(dict.fromkeys(chunk_ids))
    chunk_id = item.get("gt_chunk_id")
    return [chunk_id] if chunk_id else []


def _get_relation(item: dict) -> str:
    """null/누락 relation은 확정 스키마의 단일 근거 레코드로 처리한다."""
    relation = item.get("chunk_relation") or "single"
    if relation not in {"single", "or", "and"}:
        raise ValueError(f"{item.get('id')}: unsupported chunk_relation={relation!r}")
    return relation


def _hit_at_k(gold_ids: list[str], retrieved_ids: list[str], relation: str, k: int) -> bool:
    """single/or는 ANY, and는 ALL 조건으로 청크 hit을 판정한다."""
    if not gold_ids:
        return False
    top_k = set(retrieved_ids[:k])
    if relation in {"single", "or"}:
        return bool(set(gold_ids) & top_k)
    return set(gold_ids).issubset(top_k)


def _recalled_gold_at_k(gold_ids: list[str], retrieved_ids: list[str], k: int) -> int:
    """Top-k에 회수된 서로 다른 gold chunk 수를 반환한다."""
    return len(set(gold_ids) & set(retrieved_ids[:k]))


def _recall_at_k(gold_ids: list[str], retrieved_ids: list[str], k: int) -> float:
    """한 문항의 gold chunk coverage를 반환한다."""
    if not gold_ids:
        return 0.0
    return _recalled_gold_at_k(gold_ids, retrieved_ids, k) / len(set(gold_ids))


def _recall_at_10(gold_ids: list[str], retrieved_ids: list[str]) -> float:
    """Top-10에 회수된 gold chunk 비율(관계와 무관한 gold coverage)이다."""
    return _recall_at_k(gold_ids, retrieved_ids, RETRIEVAL_DEPTH)


def _reciprocal_rank(gold_ids: list[str], retrieved_ids: list[str], relation: str) -> float:
    """ANY는 첫 gold, AND는 마지막 필수 gold가 나온 순위의 역수다."""
    if not gold_ids:
        return 0.0
    gold_set = set(gold_ids)
    seen: set[str] = set()
    for rank, chunk_id in enumerate(retrieved_ids, 1):
        if chunk_id not in gold_set:
            continue
        if relation in {"single", "or"}:
            return 1.0 / rank
        seen.add(chunk_id)
        if seen == gold_set:
            return 1.0 / rank
    return 0.0


def _empty_metrics() -> dict:
    return {
        "n_total": 0,
        "n_evaluated": 0,
        "n_excluded_unanswerable": 0,
        "n_inconsistent": 0,
        **{f"hit@{k}": 0.0 for k in KS},
        **{f"recalled_gold@{k}": 0 for k in RECALL_KS},
        "total_gold": 0,
        **{f"recall@{k}": 0.0 for k in RECALL_KS},
        **{f"macro_recall@{k}": 0.0 for k in RECALL_KS},
        "missing_gold@10": 0,
        "mrr": 0.0,
    }


def _metric_values(gold_ids: list[str], retrieved_ids: list[str], relation: str) -> dict:
    return {
        **{f"hit@{k}": _hit_at_k(gold_ids, retrieved_ids, relation, k) for k in KS},
        **{f"recalled_gold@{k}": _recalled_gold_at_k(gold_ids, retrieved_ids, k)
           for k in RECALL_KS},
        **{f"recall@{k}": _recall_at_k(gold_ids, retrieved_ids, k)
           for k in RECALL_KS},
        # rank cutoff matches RETRIEVAL_DEPTH so MRR stays MRR@10, not MRR@len(retrieved_ids).
        "mrr": _reciprocal_rank(gold_ids, retrieved_ids[:RETRIEVAL_DEPTH], relation),
    }


def _add_metrics(metrics: dict, values: dict, gold_ids: list[str]) -> None:
    """문항 하나의 결과를 집계 메트릭에 더한다."""
    metrics["n_evaluated"] += 1
    unique_gold_count = len(set(gold_ids))
    metrics["total_gold"] += unique_gold_count
    for key in ("hit@1", "hit@3", "hit@5", "hit@10", "mrr"):
        metrics[key] += values[key]
    for k in RECALL_KS:
        metrics[f"recalled_gold@{k}"] += values[f"recalled_gold@{k}"]
        metrics[f"macro_recall@{k}"] += values[f"recall@{k}"]


def _finalize_metrics(metrics: dict) -> dict:
    """Hit/MRR/MacroRecall은 문항 기준, (micro)Recall은 gold chunk occurrence 기준으로 정규화한다."""
    if metrics["n_evaluated"]:
        denominator = metrics["n_evaluated"]
        for key in ("hit@1", "hit@3", "hit@5", "hit@10", "mrr"):
            metrics[key] /= denominator
        for k in RECALL_KS:
            metrics[f"macro_recall@{k}"] /= denominator
    if metrics["total_gold"]:
        denominator = metrics["total_gold"]
        for k in RECALL_KS:
            metrics[f"recall@{k}"] = metrics[f"recalled_gold@{k}"] / denominator
    metrics["missing_gold@10"] = metrics["total_gold"] - metrics["recalled_gold@10"]
    return metrics


def _parent_ids(
    chunk_ids: list[str],
    chunk_to_doc: dict[str, str],
    missing_prefix: str = "__missing__",
) -> list[str]:
    """알 수 없는 gold는 절대 검색에 매칭되지 않는 별도 식별자로 보존한다."""
    return list(dict.fromkeys(chunk_to_doc.get(chunk_id, f"{missing_prefix}:{chunk_id}")
                              for chunk_id in chunk_ids))


def evaluate_retrieval(
    gt_data: Iterable[dict],
    retrieved_data: Iterable[dict],
    chunk_metadata: Iterable[dict] | None = None,
) -> dict:
    """GT와 검색 결과를 결정론적으로 평가한다.

    answerable=false는 모든 Hit@K, Recall@K, MRR의 분모에서 제외된다.
    Recall은 문항 평균이 아니라 전체 gold chunk 기준으로 계산한다.
    ``chunk_metadata``를 주면 기존 방식과 같은 문서 단위 참고 지표도 함께 계산한다.
    """
    retrieved_by_id: dict[str, list[str]] = {}
    for item in retrieved_data:
        item_id = item["id"]
        if item_id in retrieved_by_id:
            raise ValueError(f"duplicate retrieved result id: {item_id}")
        retrieved_by_id[item_id] = item.get("retrieved_chunk_ids", [])

    chunk_to_doc = {item["chunk_id"]: item["parent_doc_id"]
                    for item in (chunk_metadata or [])}
    chunk_metrics = _empty_metrics()
    doc_metrics = _empty_metrics() if chunk_metadata is not None else None
    relation_metrics: dict[str, dict] = {}
    domain_metrics: dict[str, dict] = {}
    audit_rows = []

    for gt_item in gt_data:
        item_id = gt_item["id"]
        gold_ids = _get_gt_chunk_ids(gt_item)
        relation = _get_relation(gt_item)
        retrieved_ids = retrieved_by_id.get(item_id, [])
        answerable = gt_item.get("answerable") is True
        chunk_metrics["n_total"] += 1
        if doc_metrics is not None:
            doc_metrics["n_total"] += 1

        if not answerable:
            chunk_metrics["n_excluded_unanswerable"] += 1
            if doc_metrics is not None:
                doc_metrics["n_excluded_unanswerable"] += 1
            if gold_ids:
                chunk_metrics["n_inconsistent"] += 1
                if doc_metrics is not None:
                    doc_metrics["n_inconsistent"] += 1
            audit_rows.append({"id": item_id, "answerable": False, "chunk_relation": relation,
                               "gold_chunk_ids": gold_ids, "retrieved_chunk_ids": retrieved_ids,
                               "excluded": True})
            continue

        chunk_metrics["n_evaluated"] += 1
        if not gold_ids:
            chunk_metrics["n_inconsistent"] += 1
            if doc_metrics is not None:
                doc_metrics["n_inconsistent"] += 1
        values = _metric_values(gold_ids, retrieved_ids, relation)
        # n_evaluated is incremented above to preserve the explicit
        # answerable=true denominator, including an invalid no-gold label.
        chunk_metrics["total_gold"] += len(set(gold_ids))
        for key in ("hit@1", "hit@3", "hit@5", "hit@10", "mrr"):
            chunk_metrics[key] += values[key]
        for k in RECALL_KS:
            chunk_metrics[f"recalled_gold@{k}"] += values[f"recalled_gold@{k}"]
            chunk_metrics[f"macro_recall@{k}"] += values[f"recall@{k}"]

        relation_metrics.setdefault(relation, _empty_metrics())
        relation_metrics[relation]["n_total"] += 1
        _add_metrics(relation_metrics[relation], values, gold_ids)

        domain = gt_item.get("business_function") or "(unknown)"
        domain_metrics.setdefault(domain, _empty_metrics())
        domain_metrics[domain]["n_total"] += 1
        _add_metrics(domain_metrics[domain], values, gold_ids)

        row = {"id": item_id, "answerable": True, "chunk_relation": relation,
               "business_function": domain, "question": gt_item.get("question", ""),
               "gold_chunk_ids": gold_ids, "retrieved_chunk_ids": retrieved_ids,
               "excluded": False, "chunk": values}
        if doc_metrics is not None:
            gold_doc_ids = _parent_ids(gold_ids, chunk_to_doc, "__missing_gold__")
            retrieved_doc_ids = _parent_ids(retrieved_ids, chunk_to_doc, "__missing_retrieved__")
            doc_values = _metric_values(gold_doc_ids, retrieved_doc_ids, relation)
            _add_metrics(doc_metrics, doc_values, gold_doc_ids)
            row["gold_parent_doc_ids"] = gold_doc_ids
            row["doc"] = doc_values
        audit_rows.append(row)

    _finalize_metrics(chunk_metrics)
    if doc_metrics is not None:
        _finalize_metrics(doc_metrics)
    for metrics in relation_metrics.values():
        _finalize_metrics(metrics)
    for metrics in domain_metrics.values():
        _finalize_metrics(metrics)

    gold_coverage = {
        "gold_chunk_total": chunk_metrics["total_gold"],
        "recalled_gold@3": chunk_metrics["recalled_gold@3"],
        "recalled_gold@5": chunk_metrics["recalled_gold@5"],
        "recalled_gold@10": chunk_metrics["recalled_gold@10"],
        "missing_gold@10": chunk_metrics["missing_gold@10"],
        "gold_coverage@10": chunk_metrics["recall@10"],
    }

    return {
        "chunk_level": chunk_metrics,
        "document_level": doc_metrics,
        "relation_metrics": relation_metrics,
        "domain_metrics": domain_metrics,
        "gold_coverage": gold_coverage,
        "audit_rows": audit_rows,
    }


def _load_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


def _search_results(gt_data: Iterable[dict], mode: str, data_dir: Path) -> list[dict]:
    """기존 검색 구현을 그대로 호출하며 알고리즘이나 라벨은 변경하지 않는다."""
    import rag

    searcher = rag.Searcher(index_dir=str(data_dir / "index"))
    return [{"id": item["id"],
             "retrieved_chunk_ids": [hit["chunk_id"] for hit in searcher.search(
                 item["question"], k=RETRIEVAL_DEPTH, mode=mode)]}
            for item in gt_data]


def _format_ids(chunk_ids: list[str]) -> str:
    return "<br>".join(f"`{chunk_id}`" for chunk_id in chunk_ids) or "-"


def _pct_table(counter: dict, total: int, headers: tuple[str, str]) -> list[str]:
    lines = [f"| {headers[0]} | {headers[1]} |", "|---|---:|"]
    for key, count in sorted(counter.items(), key=lambda kv: -kv[1]):
        pct = count / total * 100 if total else 0.0
        lines.append(f"| {key} | {count} ({pct:.1f}%) |")
    return lines


def _error_analysis(rows: list[dict], chunk_to_bf: dict[str, str] | None) -> dict:
    """미적중(top-3 밖) 문항의 원인·혼입 패턴을 집계한다.

    chunk_to_bf가 없으면(청크 메타 미제공) 목록만 반환하고 도메인 분류는 생략한다.
    """
    misses = [row for row in rows if not row["excluded"] and not row["chunk"]["hit@3"]]
    by_domain: dict[str, int] = {}
    for row in misses:
        by_domain[row["business_function"]] = by_domain.get(row["business_function"], 0) + 1

    if chunk_to_bf is None:
        return {"misses": misses, "by_domain": by_domain, "cause": {}, "intruders": {},
                "cross_flow": {}, "worst": [row for row in misses if not row["chunk"]["hit@10"]][:8]}

    cause: dict[str, int] = {}
    intruders: dict[str, int] = {}
    cross_flow: dict[tuple[str, str], int] = {}
    for row in misses:
        top3 = row["retrieved_chunk_ids"][:3]
        top3_bf = [chunk_to_bf.get(cid, "(unknown)") for cid in top3]
        gt_bf = row["business_function"]
        reason = "정분류(업무 내 랭킹·청킹)" if gt_bf in top3_bf else "오분류(다른 업무로 혼입)"
        cause[reason] = cause.get(reason, 0) + 1
        for cid, bf in zip(top3, top3_bf):
            if bf != gt_bf:
                intruders[cid] = intruders.get(cid, 0) + 1
        if top3_bf and top3_bf[0] != gt_bf:
            key = (gt_bf, top3_bf[0])
            cross_flow[key] = cross_flow.get(key, 0) + 1

    worst = [row for row in misses if not row["chunk"]["hit@10"]][:8]
    return {"misses": misses, "by_domain": by_domain, "cause": cause,
            "intruders": intruders, "cross_flow": cross_flow, "worst": worst}


def write_report(result: dict, output_path: Path, gt_path: Path, retrieval_source: str,
                  chunk_metadata: list[dict] | None = None) -> None:
    """새 v3 결과 전용 Markdown 보고서를 저장한다."""
    chunk = result["chunk_level"]
    doc = result["document_level"]
    relation_metrics = result["relation_metrics"]
    domain_metrics = result["domain_metrics"]
    gold_coverage = result["gold_coverage"]
    rows = result["audit_rows"]
    chunk_to_bf = ({c["chunk_id"]: c.get("business_function", "(unknown)") for c in chunk_metadata}
                   if chunk_metadata else None)

    try:
        import rag
        retriever_desc = (f"RRF(k={rag.RRF_K}, alpha={rag.RRF_ALPHA}) + "
                          f"BF_MODE={rag.BF_MODE}(cap={rag.BF_DOC_CAP})")
        embedding_desc = rag.MODEL_NAME
    except Exception:
        retriever_desc = "정보 없음(외부 검색 결과 사용)"
        embedding_desc = "정보 없음(외부 검색 결과 사용)"

    lines = ["# KDIC 검색 평가 리포트 v3", "",
              "## 1. 개요", "",
              "청크 단위 검색 평가 — gold `chunk_id`를 직접 비교해 기존 `parent_doc_id` 기준 평가가",
              "안고 있던 '같은 문서면 적중' 거품을 배제하고 실제 청크 적중률을 측정한다.", "",
              "## 2. 평가 대상 / 데이터 구성", "",
              f"- GT 라벨: `{gt_path}`",
              f"- 전체 {chunk['n_total']}건 · 평가 대상(answerable=true) {chunk['n_evaluated']}건 · "
              f"제외(answerable=false) {chunk['n_excluded_unanswerable']}건 · 라벨 불일치 {chunk['n_inconsistent']}건",
              ""]
    lines += _pct_table({rel: m["n_total"] for rel, m in relation_metrics.items()},
                        chunk["n_evaluated"], ("relation", "문항수"))
    lines += ["",
              "## 3. 실험 파라미터", "",
              f"- Corpus: {len(chunk_metadata)}개 청크 (`chunks.jsonl`)" if chunk_metadata is not None
              else "- Corpus: 정보 없음(청크 메타 미제공)",
              f"- Retriever: {retriever_desc}",
              f"- Embedding: {embedding_desc}",
              f"- 검색 단위: chunk (기존 parent_doc_id 평가와 별도)",
              f"- 검색 소스: {retrieval_source}",
              f"- Top-K: {RETRIEVAL_DEPTH}", "",
              "## 4. 평가 지표 및 판정 기준", "",
              "- Hit@1/3/5/10: Top-K 안에 gold가 relation 조건을 만족하며 존재하는지 (분모: 평가 대상 문항수).",
              "- Macro Recall@3/5/10: 문항별 recall(Top-K에서 회수한 gold 비율)의 문항 평균. 주지표.",
              "- (참고) Micro Recall@3/5/10: 전체 gold occurrence 기준 합산 recall. 문항당 gold 개수가 많을수록 가중치가 커짐 — 보조지표로만 사용.",
              "- Gold Coverage: 전체 gold chunk occurrence 중 Top-10에서 회수되지 못한 비율(`missing_gold@10`).",
              "- MRR: relation 조건을 만족하는 gold가 처음 나온 순위의 역수, Top-10까지만 본다(MRR@10).",
              "- relation 판정: `single`/`or`는 ANY(하나만 맞아도 적중), `and`는 ALL(전부 맞아야 적중); null/누락은 single.",
              "- answerable=false 문항은 Hit@K·Recall@K·MRR 분모에서 제외.", "",
              "## 5. 전체 평가 결과", "",
              "| Hit@1 | Hit@3 | Hit@5 | Hit@10 | MRR |",
              "|---:|---:|---:|---:|---:|",
              f"| {chunk['hit@1']:.3f} | {chunk['hit@3']:.3f} | {chunk['hit@5']:.3f} | {chunk['hit@10']:.3f} | {chunk['mrr']:.3f} |",
              "",
              "| MacroRecall@3 | MacroRecall@5 | MacroRecall@10 |",
              "|---:|---:|---:|",
              f"| {chunk['macro_recall@3']:.3f} | {chunk['macro_recall@5']:.3f} | {chunk['macro_recall@10']:.3f} |",
              "",
              f"- (참고) Micro Recall@10: {chunk['recall@10']:.3f} · gold 총계(occurrence): "
              f"{gold_coverage['gold_chunk_total']} · 미회수(@10): {gold_coverage['missing_gold@10']}", "",
              "## 6. Relation별 결과", "",
              "| relation | 문항수 | Hit@1 | Hit@3 | Hit@5 | MacroRecall@10 | MRR |",
              "|---|---:|---:|---:|---:|---:|---:|"]
    for rel in ("single", "or", "and"):
        m = relation_metrics.get(rel)
        if m:
            lines.append(f"| {rel} | {m['n_total']} | {m['hit@1']:.3f} | {m['hit@3']:.3f} | "
                        f"{m['hit@5']:.3f} | {m['macro_recall@10']:.3f} | {m['mrr']:.3f} |")

    lines += ["", "## 7. 도메인별 결과", "",
              "| 업무 | 문항수 | Hit@1 | Hit@3 | Hit@5 | MacroRecall@10 | MRR |",
              "|---|---:|---:|---:|---:|---:|---:|"]
    for domain, m in sorted(domain_metrics.items(), key=lambda kv: -kv[1]["n_total"]):
        lines.append(f"| {domain} | {m['n_total']} | {m['hit@1']:.3f} | {m['hit@3']:.3f} | "
                    f"{m['hit@5']:.3f} | {m['macro_recall@10']:.3f} | {m['mrr']:.3f} |")

    err = _error_analysis(rows, chunk_to_bf)
    n_miss = len(err["misses"])
    lines += ["", "## 8. 오류 분석", "",
              f"- 미적중(top-3 밖) {n_miss}건 / 평가 대상 {chunk['n_evaluated']}건 "
              f"({n_miss / chunk['n_evaluated'] * 100 if chunk['n_evaluated'] else 0:.1f}%)", ""]
    if err["cause"]:
        lines += ["### 8-1. 미적중 원인 분포", "",
                  "정분류: gt 업무 청크가 top-3 안에 있으나 순위/청킹 문제로 놓침. "
                  "오분류: top-3 전부 다른 업무 청크로 채워짐. "
                  "(업무 분류기가 적용됐는지까지는 이 표만으로 알 수 없음 — 세부 조사는 수작업 검토 필요.)", ""]
        lines += _pct_table(err["cause"], n_miss, ("원인", "건수"))
    else:
        lines += ["_청크 메타(`--chunks`)가 없어 원인 분류를 생략함._"]

    lines += ["", "### 8-2. 업무별 미적중", "",
              "| 업무 | 문항수 | 미적중 | 미적중률 |", "|---|---:|---:|---:|"]
    for domain, count in sorted(err["by_domain"].items(), key=lambda kv: -kv[1]):
        total = domain_metrics[domain]["n_total"]
        lines.append(f"| {domain} | {total} | {count} | {count / total * 100 if total else 0:.1f}% |")

    if err["intruders"]:
        lines += ["", "### 8-3. 혼입 문서 랭킹 (미적중 top-3를 잠식한 다른 업무 청크)", "",
                  "| 청크 ID | 출현 | 업무 |", "|---|---:|---|"]
        for cid, count in sorted(err["intruders"].items(), key=lambda kv: -kv[1])[:15]:
            lines.append(f"| `{cid}` | {count} | {chunk_to_bf.get(cid, '(unknown)')} |")

    if err["cross_flow"]:
        lines += ["", "### 8-4. 업무 간 혼입 흐름 (정답 업무 → top-1 업무)", "",
                  "| 정답 업무 | → top-1 업무 | 건수 |", "|---|---|---:|"]
        for (gt_bf, top1_bf), count in sorted(err["cross_flow"].items(), key=lambda kv: -kv[1]):
            lines.append(f"| {gt_bf} | {top1_bf} | {count} |")

    if err["worst"]:
        lines += ["", "### 8-5. 대표 실패 사례 (gold가 top-10에도 없음)", ""]
        for row in err["worst"]:
            lines += [f"- **[{row['business_function']}] {row['id']}** — {row['question']}",
                     f"  - gold: {_format_ids(row['gold_chunk_ids'])}",
                     f"  - top-3: {_format_ids(row['retrieved_chunk_ids'][:3])}"]

    lines += ["", "## 9. 개선 우선순위", ""]
    priorities = []
    for domain, count in sorted(err["by_domain"].items(), key=lambda kv: -kv[1])[:3]:
        total = domain_metrics[domain]["n_total"]
        priorities.append(f"'{domain}' 업무 — 미적중 {count}/{total}건"
                          f"({count / total * 100 if total else 0:.1f}%)로 가장 취약함")
    and_m = relation_metrics.get("and")
    if and_m and and_m["hit@3"] < chunk["hit@3"]:
        priorities.append(f"`and` relation(다중 근거 필요) Hit@3={and_m['hit@3']:.3f}로 "
                          f"전체 평균({chunk['hit@3']:.3f})보다 낮음 — 다중 청크 회수 전략 점검 필요")
    if err["intruders"]:
        top_intruder = max(err["intruders"].items(), key=lambda kv: kv[1])
        priorities.append(f"`{top_intruder[0]}` 청크가 미적중 {top_intruder[1]}건에서 top-3를 잠식 "
                          f"— 업무 분류/부스트 파라미터 재검토 대상 1순위")
    lines += [f"{i}. {p}" for i, p in enumerate(priorities, 1)] or ["_미적중 사례 없음._"]

    if doc is not None:
        lines += ["", "## 10. 참고", "",
                  "기존 document-level(같은 `parent_doc_id`면 적중) 판정과 비교한 거품(percentage points):", "",
                  "| 지표 | chunk-level | document-level | 문서 단위 거품 |",
                  "|---|---:|---:|---:|"]
        for key in ("hit@1", "hit@3", "hit@5", "recall@10", "mrr"):
            bubble = (doc[key] - chunk[key]) * 100
            lines.append(f"| {key} | {chunk[key]:.3f} | {doc[key]:.3f} | {bubble:+.1f}pp |")

    lines += ["", "## 부록. 문항별 라벨과 검색 청크", "",
              "| id | answerable | relation | gold chunk IDs | Top-10 retrieved chunk IDs | chunk Hit@1/3/5/10 | doc Hit@1/3/5/10 |",
              "|---|---|---|---|---|---|---|"]
    for row in rows:
        if row["excluded"]:
            chunk_hits = doc_hits = "excluded"
        else:
            chunk_hits = "/".join(str(int(row["chunk"][f"hit@{k}"])) for k in KS)
            doc_hits = ("/".join(str(int(row["doc"][f"hit@{k}"])) for k in KS)
                        if "doc" in row else "-")
        lines.append("| {id} | {answerable} | {relation} | {gold} | {retrieved} | {chunk_hits} | {doc_hits} |".format(
            id=row["id"], answerable=str(row["answerable"]).lower(), relation=row["chunk_relation"],
            gold=_format_ids(row["gold_chunk_ids"]),
            retrieved=_format_ids(row["retrieved_chunk_ids"][:RETRIEVAL_DEPTH]),
            chunk_hits=chunk_hits, doc_hits=doc_hits))

    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def _print_group(title: str, rows: list[tuple[str, str]]) -> None:
    """제목 아래 key: value를 한 줄씩 나열하고, 라벨 너비를 맞춘 표 느낌으로 출력한다."""
    print(f"[{title}]")
    label_width = max(len(label) for label, _ in rows)
    for label, value in rows:
        print(f"  {label.ljust(label_width)} : {value}")
    print()  # 다음 그룹과 빈 줄 하나(연속 줄바꿈 두 번) 간격.


def _print_summary(result: dict, report_path: Path) -> None:
    """콘솔 요약을 항목마다 줄을 나누고, 그룹 사이는 빈 줄로 구분해 가독성 있게 출력한다."""
    m = result["chunk_level"]
    print("=== KDIC v3 평가 결과 ===")
    print()
    _print_group("데이터", [
        ("n_total", str(m["n_total"])),
        ("n_evaluated", str(m["n_evaluated"])),
        ("n_excluded", str(m["n_excluded_unanswerable"])),
        ("n_inconsistent", str(m["n_inconsistent"])),
    ])
    _print_group("Hit@K", [
        ("Hit@1", f"{m['hit@1']:.3f}"),
        ("Hit@3", f"{m['hit@3']:.3f}"),
        ("Hit@5", f"{m['hit@5']:.3f}"),
        ("Hit@10", f"{m['hit@10']:.3f}"),
    ])
    _print_group("MacroRecall", [
        ("@3", f"{m['macro_recall@3']:.3f}"),
        ("@5", f"{m['macro_recall@5']:.3f}"),
        ("@10", f"{m['macro_recall@10']:.3f}  (micro@10={m['recall@10']:.3f})"),
    ])
    _print_group("MRR", [("MRR@10", f"{m['mrr']:.3f}")])
    print(f"report -> {report_path}")


def main() -> None:
    data_dir = _default_data_dir()
    parser = argparse.ArgumentParser(description="KDIC chunk-level retrieval evaluation v3")
    parser.add_argument("--gt", default=data_dir / "gt_relabeled_v7.jsonl", type=Path)
    parser.add_argument("--chunks", default=data_dir / "chunks.jsonl", type=Path)
    parser.add_argument("--retrieved", type=Path,
                        help="id, retrieved_chunk_ids JSONL. 생략하면 현재 rag.Searcher로 검색한다.")
    parser.add_argument("--mode", default="hybrid_bf", help="--retrieved 생략 시 rag 검색 mode")
    parser.add_argument("--report", default=data_dir / "eval_anchor_v3_report.md", type=Path)
    parser.add_argument("--limit", type=int, default=0,
                        help="처음 N건만 평가(스모크 테스트/속도 측정용, 대표성 있는 지표가 아님). 0이면 전체.")
    args = parser.parse_args()

    gt_data = _load_jsonl(args.gt)
    if args.limit:
        gt_data = gt_data[: args.limit]
    if args.retrieved:
        retrieved_data = _load_jsonl(args.retrieved)
        retrieval_source = f"외부 결과 `{args.retrieved}` (검색 미실행)"
    else:
        retrieved_data = _search_results(gt_data, args.mode, data_dir)
        retrieval_source = f"`rag.Searcher.search(question, k=10, mode={args.mode!r})`"
    chunk_metadata = _load_jsonl(args.chunks)
    result = evaluate_retrieval(gt_data, retrieved_data, chunk_metadata)
    args.report.parent.mkdir(parents=True, exist_ok=True)
    write_report(result, args.report, args.gt, retrieval_source, chunk_metadata)
    _print_summary(result, args.report)


if __name__ == "__main__":
    main()


## 4. 데이터 업로드 — **직접 파일 선택 필요**

로컬의 `crawler/data/chunks.jsonl`과 `crawler/data/gt_relabeled_v7.jsonl` **두 파일을 한 번에 선택**해서 업로드하세요.

In [ ]:
from google.colab import files
uploaded = files.upload()  # chunks.jsonl, gt_relabeled_v7.jsonl 두 개 선택

import shutil
for name in uploaded:
    shutil.move(name, f"data/{name}")
print("업로드된 파일:", list(uploaded.keys()))

assert os.path.isfile("data/chunks.jsonl"), "chunks.jsonl이 안 올라왔습니다."
assert os.path.isfile("data/gt_relabeled_v7.jsonl"), "gt_relabeled_v7.jsonl이 안 올라왔습니다."
print("ok")


## 5. 인덱스 빌드 (`chunks.jsonl` → FAISS + BM25)

로컬 인덱스 파일을 옮기는 대신, 같은 `rag.py` 코드(`Store`)로 Colab에서 새로 빌드합니다.
처음 실행 시 임베딩 모델(`dragonkue/BGE-m3-ko`)을 허깅페이스에서 내려받습니다(수백MB, 인터넷 필요).

In [ ]:
import json, importlib
import rag
importlib.reload(rag)

chunks = [json.loads(l) for l in open("data/chunks.jsonl", encoding="utf-8") if l.strip()]
print(f"{len(chunks)}개 청크 로드")

store = rag.Store(index_dir="data/index")
store.chunks = chunks
store.embs = rag.embed_texts([rag.chunk_embed_text(c) for c in chunks])
store.build_and_save()
print("인덱스 빌드 완료 ->", os.listdir("data/index"))


## 6. 베이스라인 평가 (`hybrid_bf`, 리랭커 없이) — 비교 기준

In [ ]:
!python eval_anchor_v3.py --mode hybrid_bf --report data/eval_report_baseline.md


## 7. 리랭커 적용 평가 (`hybrid_bf_rerank`)

처음 실행 시 `jinaai/jina-reranker-v2-base-multilingual`을 허깅페이스에서 내려받습니다(`trust_remote_code=True`로 Jina가 배포한 커스텀 모델 코드를 실행함 — 신뢰할 수 있는 공개 모델이지만 참고해주세요).

### 7-1. 스모크 테스트 — 먼저 30건만 돌려서 속도부터 확인

`--limit 30`은 GT의 앞쪽 30건만 평가합니다. **업무가 뒤섞이지 않은 순서라 지표 자체(Hit@K 등)는 대표성이 없습니다** — 이 셀의 목적은 "1문항당 몇 초 걸리는지" 시간만 재는 것입니다. 아래 `%%time`으로 걸린 시간을 보고 전체 1251건이 얼마나 걸릴지 가늠하세요(예: 30건에 1분이면 1251건은 약 42분).

In [ ]:
%%time
!python eval_anchor_v3.py --mode hybrid_bf_rerank --limit 30 --report data/eval_report_rerank_smoke.md


### 7-2. 전체 실행

스모크 테스트로 시간 감을 잡았으면, 전체 1251건을 돌립니다(시간이 꽤 걸릴 수 있으니 감안하고 실행하세요).

In [ ]:
!python eval_anchor_v3.py --mode hybrid_bf_rerank --report data/eval_report_rerank.md


## 8. 결과 비교 (Hit@K / MacroRecall 요약만 추출)

In [ ]:
def summary(path):
    text = open(path, encoding="utf-8").read()
    start = text.find("## 5. 전체 평가 결과")
    end = text.find("## 6. Relation별 결과")
    if start == -1 or end == -1:
        return "(파싱 실패 — 리포트 파일을 직접 확인하세요)"
    return text[start:end].strip()

print("=== 베이스라인 (hybrid_bf) ===")
print(summary("data/eval_report_baseline.md"))
print()
print("=== 리랭커 적용 (hybrid_bf_rerank) ===")
print(summary("data/eval_report_rerank.md"))


## 9. 리포트 파일 다운로드 (선택)

In [ ]:
from google.colab import files as _files
_files.download("data/eval_report_baseline.md")
_files.download("data/eval_report_rerank.md")
